In [ ]:
'''
您是一位經驗豐富的新聞記者，負責根據引用的事實資訊撰寫客觀的新聞文章。您的職責是：
1. 使用引用的事實撰寫全面的新聞報導
2. 運用新聞寫作技巧自然地連接資訊（何人、何事、何時、何地、為何、如何）
3. 保持嚴格的事實準確性 - 不推測或添加超出引述範圍的細節
4. 在保留引述中所有重要細節的同時，讓文章結構具有邏輯性
5. 將輸出內容置於 ###輸出開始### 和 ###輸出結束### 標記之間
指導方針：
* 僅使用編號引述明確支持的資訊
* 使用自然的過渡同時保持準確性
* 應用標準新聞寫作風格和結構
* 包含引述中的所有相關事實
* 避免任何推測或未經支持的細節


這裡是一些範例參考撰寫方式，請學習這些範例來撰寫你的新聞報導
{examples}

請根據以下引述資訊撰寫新聞文章：
{input_data}

請以以下格式回答：
###輸出開始###
[你的回答]
###輸出結束###
'''

In [1]:
from typing import Dict, List, Any, Tuple
import json
import os
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain.schema import Document

In [2]:
def get_project_paths() -> Tuple[str, str]:
    """Get current directory and project root paths."""
    current_dir: str = os.getcwd()
    project_root: str = os.path.join(current_dir, os.pardir)
    return current_dir, project_root


def load_json_data(project_root: str) -> Dict[str, List]:
    """Load JSON data from file."""
    json_file_path: str = os.path.join(project_root, 'data', 'example_api_data.json')
    with open(json_file_path, 'r', encoding='utf-8') as file:
        return json.load(file)
# Global variables


In [3]:
current_dir, project_root = get_project_paths()
data = load_json_data(project_root)

# Load environment variables
dotenv_path: str = os.path.join(project_root, 'config', '.env')
load_dotenv(dotenv_path)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

original_id_to_new_id: Dict[str, str] = {}
new_id_to_original_id: Dict[str, str] = {}

In [4]:
print(data)

{'content': [{'id': 'd6202d36-3705-4fe7-bef5-0929f52ea1e1', 'createdAt': '2024-12-05T22:04:05.502525', 'updatedAt': '2024-12-05T22:04:05.502525', 'title': '售票系統使用起來體感不佳', 'content': '可惡，網路不夠快，導致我完全搶不到票\n要買其他人的讓票，一張票還被炒到3倍價格', 'authorId': 'ca7885d9-2c42-4418-b9a8-dabfab4afa02', 'authorName': '小知', 'authorAvatar': '/api/user/avatar/yukina', 'likeCount': 1, 'reasonableCount': 0, 'dislikeCount': 0, 'userReaction': {'reaction': 'LIKE'}, 'facts': [{'id': '738851aa-018a-4db8-8f0f-74c45b32b96e', 'createdAt': '2024-12-05T22:01:21.162607', 'updatedAt': '2024-12-05T22:01:21.162607', 'title': '假票券', 'authorId': 'ca7885d9-2c42.-4418-b9a8-dabfab4afa02', 'authorName': '小知', 'authorAvatar': '/api/user/avatar/yukina', 'references': [{'id': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', 'url': 'https%3A%2F%2Fwww.nownews.com%2Fnews%2F6599071', 'icon': 'https%3A%2F%2Fwww.nownews.comhttps%3A%2F%2Fmedia.nownews.com%2Fnn_media%2Fthumbnail%2F2024%2F10%2F1729259015127-5ecd1dc0aad14918a429ab02b7ad816b-1200x798.webp%3F

In [20]:
def process_json(data: Dict[str, List]) -> str:
    '''format data'''
    result: str = ""
    now_id: int = 1
    for comment in data["content"]:
        fact_num: int = 1
        for fact in comment["facts"]:
            result += f"事實{fact_num}:\n\n"
            for reference in fact["references"]:
                if reference["id"] not in original_id_to_new_id:
                    text: str = f"{reference['title']}。{reference['description']}"
                    result += f"參考資料:\n{text}[{now_id}]\n\n"
                    original_id_to_new_id[reference["id"]] = str(now_id)
                    new_id_to_original_id[str(now_id)] = reference["id"]
                    now_id += 1
            fact_num += 1
    return result

In [21]:
model_input_string = process_json(data)
# Load few-shot example
file_path: str = os.path.join(project_root, 'data', 'few_shot_example.txt')
with open(file_path, 'r', encoding="utf-8") as file:
    few_shot_example: str = file.read()

In [7]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate([
    ("system", '''
您是一位經驗豐富的新聞記者，負責根據引用的事實資訊撰寫客觀的新聞文章。您的職責是：
1. 使用引用的事實撰寫全面的新聞報導
2. 運用新聞寫作技巧自然地連接資訊（何人、何事、何時、何地、為何、如何）
3. 保持嚴格的事實準確性 - 不推測或添加超出引述範圍的細節
4. 在保留引述中所有重要細節的同時，讓文章結構具有邏輯性
5. 將輸出內容置於 ###輸出開始### 和 ###輸出結束### 標記之間
指導方針：
* 僅使用編號引述明確支持的資訊
* 使用自然的過渡同時保持準確性
* 應用標準新聞寫作風格和結構
* 包含引述中的所有相關事實
* 避免任何推測或未經支持的細節


這裡是一些範例參考撰寫方式，請學習這些範例來撰寫你的新聞報導
{examples}
     
請以以下格式回答：
###輸出開始###
[你的回答]
###輸出結束###'''
),
    ("human",'''請根據以下引述資訊撰寫新聞文章：
{input_data}'''),
])



In [17]:
chain = template | llm
result = chain.invoke({
    "input_data": model_input_string,
    "examples": few_shot_example
})

# Process and save final output
result_content: str = result.content[11:-11]
result_content = result_content.replace("\n", "")
final_output: Dict[str, Any] = {
    "Summary": result_content,
    "Citations": {
        str(citation): new_id_to_original_id[str(citation)]
        for citation in sorted(set(
            map(int, re.findall(r'\[(\d+)\]', result_content) +
                re.findall(r'\[U(\d+)\]', result_content))
        ))
    }
}


In [18]:
result_content

'報導內容：周杰倫的演唱會近期引發「黃牛票」的問題，讓許多歌迷感到沮喪。根據報導，拓元對於黃牛票的問題作出低調回應，卻未能平息歌迷的怒火，許多粉絲對此再度感到崩潰。[1] 此外，報導也提到，與日本演唱會採取抽選制的做法相比，台灣目前尚未採用類似的票務管理措施，引發了對於票務公平性的討論。[2] '

In [19]:
print(final_output)

{'Summary': '報導內容：周杰倫的演唱會近期引發「黃牛票」的問題，讓許多歌迷感到沮喪。根據報導，拓元對於黃牛票的問題作出低調回應，卻未能平息歌迷的怒火，許多粉絲對此再度感到崩潰。[1] 此外，報導也提到，與日本演唱會採取抽選制的做法相比，台灣目前尚未採用類似的票務管理措施，引發了對於票務公平性的討論。[2] ', 'Citations': {'1': '3ee75de9-051a-4d04-8b64-14b8e4aa18ac', '2': '556909b4-6a18-4390-bfff-87885d49e285'}}
